In [ ]:
!pip install rank_bm25 faiss-cpu sentence-transformers numpy pyngrok streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 60.9 MB/s eta 0:00:00


In [ ]:
!gdown 1Yy4n0eKUfeDYkMmGgqGYfH7ZNM3cHBVV
!gdown 1769eRWIQLB80wjB9Nh9M9Q71ra9ytyMY

Downloading...
From: https://drive.google.com/uc?id=1UVPMXmDDEIzzq6yTHMOCuJSAquwU70uf
To: /content/ewu_complete_dataset.json
100% 1.10M/1.10M [00:00<00:00, 127MB/s]
Downloading...
From: https://drive.google.com/uc?id=1jXCxcRizNrkX6g9EUAJ3Zo4XdE6xm6kj
To: /content/ewu_faculty_scraped_data_only.json
100% 170k/170k [00:00<00:00, 77.3MB/s]


In [ ]:
# ============================================
# CELL 2: Upload Data Files
# ============================================
from google.colab import files
import os
import shutil

print("=" * 60)
print("📤 UPLOAD YOUR DATA FILES")
print("=" * 60)
print("\nPlease upload the following files:")
print("   • ewu_complete_dataset(1).json")
print("   • ewu_faculty_scraped_data_only.json")
print("\nClick 'Choose Files' below and select your JSON files...\n")

uploaded = files.upload()

if not uploaded:
    print("\n❌ No files uploaded! Please run this cell again.")
else:
    print(f"\n✅ Successfully uploaded {len(uploaded)} file(s):")
    for filename in uploaded.keys():
        file_size = os.path.getsize(filename) / 1024  # KB
        print(f"   • {filename} ({file_size:.2f} KB)")
    print("\n✓ Ready to proceed!")

📤 UPLOAD YOUR DATA FILES

Please upload the following files:
   • ewu_complete_dataset(1).json
   • ewu_faculty_scraped_data_only.json

Click 'Choose Files' below and select your JSON files...



Saving ewu_faculty_scraped_data_only.json to ewu_faculty_scraped_data_only.json
Saving ewu_complete_dataset.json to ewu_complete_dataset.json

✅ Successfully uploaded 2 file(s):
   • ewu_faculty_scraped_data_only.json (166.31 KB)
   • ewu_complete_dataset.json (1070.56 KB)

✓ Ready to proceed!


In [ ]:
%%writefile /content/app.py
import streamlit as st
import pandas as pd
import numpy as np
import time
import plotly.express as px
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import json
import os
import glob

st.set_page_config(
    page_title="🎓 EWU Hybrid Search Engine",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS
st.markdown("""
<style>
    .main-header {
        font-size: 2.5rem;
        font-weight: 700;
        color: #1f77b4;
        text-align: center;
        margin-bottom: 0.5rem;
    }
    .sub-header {
        font-size: 1.5rem;
        color: #666;
        text-align: center;
        margin-bottom: 2rem;
    }
</style>
""", unsafe_allow_html=True)

def find_data_files():
    """Automatically find JSON data files in /content/."""
    json_files = glob.glob("/content/*.json")
    return json_files

@st.cache_resource
def load_engine():
    """Load and prepare the hybrid search engine."""
    data_paths = find_data_files()
    if not data_paths:
        return None, None, None, None, None, None, []

    all_records = []
    progress_bar = st.progress(0)
    status_text = st.empty()

    for idx, path in enumerate(data_paths):
        try:
            status_text.text(f"📖 Loading {os.path.basename(path)}...")
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            if isinstance(data, list):
                for item in data:
                    if isinstance(item, dict):
                        text = (
                            item.get('content') or
                            item.get('text') or
                            item.get('body') or
                            item.get('description') or
                            item.get('name', '') + ' ' + item.get('role', '') or
                            str(item)
                        )
                        all_records.append({
                            'text': str(text).strip(),
                            'source': item.get('source', os.path.basename(path)),
                            'metadata': item.get('metadata', {}),
                        })
            progress_bar.progress((idx + 1) / len(data_paths))
        except Exception as e:
            st.warning(f"⚠️ Error loading {path}: {str(e)}")
            continue

    status_text.empty()
    progress_bar.empty()

    if not all_records:
        return None, None, None, None, None, None, []

    df = pd.DataFrame(all_records)
    df['text'] = df['text'].astype(str)
    df = df[df['text'].str.strip().astype(bool)]
    df = df[df['text'].str.len() > 10]

    texts = df['text'].tolist()
    documents = df.to_dict(orient="records")

    with st.spinner(" Loading AI model..."):
        model = SentenceTransformer("all-MiniLM-L6-v2")

    with st.spinner(" Generating embeddings..."):
        embeddings = model.encode(texts, show_progress_bar=False).astype("float32")

    with st.spinner(" Building search index..."):
        dim = embeddings.shape[1]
        hnsw = faiss.IndexHNSWFlat(dim, 32)
        hnsw.add(embeddings)

    with st.spinner(" Building BM25 index..."):
        tokenized = [t.lower().split() for t in texts]
        bm25 = BM25Okapi(tokenized)

    return documents, texts, embeddings, hnsw, bm25, model, data_paths

# --- UI Header ---
st.markdown('<p class="main-header">🎓 East West University</p>', unsafe_allow_html=True)
st.markdown('<p class="sub-header">Hybrid Information Retrieval System</p>', unsafe_allow_html=True)

data_files = find_data_files()
if not data_files:
    st.error(" No data files found!")
    st.stop()

with st.spinner("⚙️ Initializing search engine..."):
    result = load_engine()

if result[0] is None:
    st.error(" Failed to load search engine.")
    st.stop()

documents, texts, doc_vectors, hnsw_index, bm25, model, loaded_files = result

# --- Sidebar ---
st.sidebar.title("⚙️ Configuration")
top_k = st.sidebar.slider("Top K Results", 3, 20, 5)
ef_search = st.sidebar.slider("HNSW efSearch", 10, 200, 50)
show_report = st.sidebar.checkbox("Show Performance Report", value=True)
show_scores = st.sidebar.checkbox("Show Relevance Scores", value=False)

# --- Search Interface ---
query = st.text_input("🔍 Enter your search query:", placeholder="e.g., CSE488 prerequisites, faculty list...")

if query:
    hnsw_index.hnsw.efSearch = ef_search
    tab1, tab2, tab3 = st.tabs([" Results", " Performance", " Details"])

    with tab1:
        with st.spinner("🔍 Searching..."):
            # 1. BM25 (Lexical)
            t0 = time.time()
            q_tokens = query.lower().split()
            bm25_scores = bm25.get_scores(q_tokens)
            bm25_idx = np.argsort(-bm25_scores)[:top_k]
            t_bm25 = (time.time() - t0) * 1000

            # 2. HNSW (Semantic)
            t0 = time.time()
            q_vec = model.encode([query]).astype("float32")
            hnsw_distances, hnsw_q_idx = hnsw_index.search(q_vec, top_k)
            t_hnsw = (time.time() - t0) * 1000

            # 3. Teleport-Walk (Proposed Hybrid)
            t0 = time.time()
            seed_doc = int(np.argmax(bm25_scores))
            seed_vec = np.array(doc_vectors[seed_doc]).astype("float32").reshape(1, -1)
            teleport_distances, teleport_idx = hnsw_index.search(seed_vec, top_k)
            t_teleport = (time.time() - t0) * 1000

        st.markdown(f"### 🎯 Top {top_k} Results (Teleport-Walk Method)")
        for i, idx in enumerate(teleport_idx[0], 1):
            doc = documents[idx]
            display_text = doc['text'][:500] + "..." if len(doc['text']) > 500 else doc['text']
            with st.expander(f"📄 Result {i}" + (f" (Confidence: {1/teleport_distances[0][i-1]:.3f})" if show_scores else "")):
                st.markdown(display_text)
                if doc.get('metadata'): st.json(doc['metadata'])

    with tab2:
        if show_report:
            bm25_set, hnsw_set, teleport_set = set(bm25_idx), set(hnsw_q_idx[0]), set(teleport_idx[0])

            report_data = [
                {"Method": "BM25 Only (Lexical)", "Time (ms)": t_bm25, "Recall vs BM25": 1.0},
                {"Method": "HNSW (Pure Semantic)", "Time (ms)": t_hnsw, "Recall vs BM25": len(hnsw_set & bm25_set)/top_k},
                {"Method": "Teleport-Walk (Hybrid) ⭐", "Time (ms)": t_teleport, "Recall vs BM25": len(teleport_set & bm25_set)/top_k},
            ]
            report = pd.DataFrame(report_data)
            st.dataframe(report.style.highlight_max(axis=0, subset=['Recall vs BM25']), use_container_width=True)

            col1, col2 = st.columns(2)
            with col1:
                st.plotly_chart(px.scatter(report, x="Time (ms)", y="Recall vs BM25", color="Method", title="Accuracy vs Speed"), use_container_width=True)
            with col2:
                st.plotly_chart(px.bar(report, x="Method", y="Time (ms)", title="Latency Comparison"), use_container_width=True)

            st.info("**Research Insight:** Teleport-Walk uses lexical anchors (BM25) to find a high-confidence neighborhood, then 'walks' the HNSW graph to capture semantic context that raw query vectors often miss.")

    with tab3:
        st.markdown("### 🔬 Comparison Breakdown")
        col1, col2, col3 = st.columns(3)
        with col1:
            st.write("**BM25 Top Hits**")
            for i in bm25_idx[:3]: st.caption(texts[i][:150] + "...")
        with col2:
            st.write("**HNSW Top Hits**")
            for i in hnsw_q_idx[0][:3]: st.caption(texts[i][:150] + "...")
        with col3:
            st.write("**Teleport-Walk Hits**")
            for i in teleport_idx[0][:3]: st.caption(texts[i][:150] + "...")

st.markdown("---")
st.markdown("<div style='text-align: center; color: #666;'>🎓 EWU Hybrid IR System | Powered by BM25 + FAISS</div>", unsafe_allow_html=True)

Overwriting /content/app.py


In [ ]:
from pyngrok import ngrok
import subprocess
import time
import os

# Set ngrok auth token
ngrok.set_auth_token("3591ju6dqcsTiwjaNiCwKJSdJMf_3RudeoETcjnosbLn7RQDR")

# Kill existing tunnels
ngrok.kill()

# Start Streamlit
subprocess.Popen([
    "streamlit", "run", "/content/app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0"
])

time.sleep(5)

# Create ngrok tunnel
public_url = ngrok.connect(8501)
print("✅ PUBLIC URL:", public_url.public_url)
print("\n⚠️ SECURITY WARNING: This ngrok token is exposed. Revoke it after use!")

✅ PUBLIC URL: https://foggiest-commercially-lane.ngrok-free.dev

⚠️ SECURITY WARNING: This ngrok token is exposed. Revoke it after use!
